***** UNIT 5 CODE *****

The goal of this code is to use "Topic Modeling" to identify common themes in a collection of text.

In [75]:
path = '/content/drive/My Drive/CIS617_TextAnalytics/TextAnalyticsCode_FordKCAP'

import pandas as pd
reviews_df = pd.read_csv(path+'/reviews.csv')
#reviews_df.head(3)

In [76]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import LatentDirichletAllocation as LDA
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [77]:
##### USING 30 "MOST NEGATIVE" REVIEWS #####
corpus = reviews_df.sort_values(by = 'rating', ascending = True)['snippet'][:30]

mystopwords = ['place','get','go','back','ford','dont','like','still','bought','2018','way','f150'] #Stop words for analyzing "All" reviews
#mystopwords = ['good','great','nice','best','awesome', 'love','place','get','go','back','ford','dont','friendly','like','still','bought','2018','way','f150'] #Stop words for analyzing "Negative Sentiment" reviews

#vectorize corpus
count_vect = CountVectorizer(stop_words=stopwords.words('english')+mystopwords, lowercase=True)
x_counts = count_vect.fit_transform(corpus)
feature_names = count_vect.get_feature_names_out()

In [78]:
#calcuatiing term-frequency times inverse document-frequency (tf-idf)
tfidf_transformer = TfidfTransformer()
x_tfidf = tfidf_transformer.fit_transform(x_counts)
x_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 819 stored elements and shape (30, 522)>

In [79]:
dimension = 5
number_of_words_per_theme = 5

#identifying themes
lda = LDA(n_components = dimension)
lda_array = lda.fit_transform(x_tfidf)

components = [lda.components_[i] for i in range(len(lda.components_))]
important_words = [sorted(feature_names, key = lambda x: components[j][np.where(feature_names == x)], reverse = True)[:number_of_words_per_theme] for j in range(len(components))]
#print(important_words)
important_words

[['one', 'window', 'loud', 'staff', '1st'],
 ['broker', 'trailer', 'truck', 'company', 'days'],
 ['even', 'security', 'workers', 'help', 'treat'],
 ['waiting', 'rude', 'full', 'rows', 'completely'],
 ['time', 'truckers', 'waste', 'expedited', 'money']]

In [80]:
##### USING "ALL" REVIEWS #####
corpus = reviews_df['snippet']

mystopwords = ['good','great','nice','best','awesome', 'love','place','get','go','back','ford','dont','friendly','like','still','bought','2018','way','f150']

#vectorize corpus
count_vect = CountVectorizer(stop_words=stopwords.words('english')+mystopwords, lowercase=True)
x_counts = count_vect.fit_transform(corpus)
feature_names = count_vect.get_feature_names_out()

In [81]:
#calcuatiing term-frequency times inverse document-frequency (tf-idf)
tfidf_transformer = TfidfTransformer()
x_tfidf = tfidf_transformer.fit_transform(x_counts)
x_tfidf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 3088 stored elements and shape (198, 1307)>

In [82]:
dimension = 5
number_of_words_per_theme = 5

#identifying themes
lda = LDA(n_components = dimension)
lda_array = lda.fit_transform(x_tfidf)

components = [lda.components_[i] for i in range(len(lda.components_))]
important_words = [sorted(feature_names, key = lambda x: components[j][np.where(feature_names == x)], reverse = True)[:number_of_words_per_theme] for j in range(len(components))]
#print(important_words)
important_words

[['drop', 'trailer', 'pm', 'trucks', 'worst'],
 ['trailer', 'staff', 'drop', 'facility', 'night'],
 ['time', 'treat', 'company', 'truck', 'workers'],
 ['delivery', 'time', 'rude', 'guard', 'lady'],
 ['rude', 'drop', 'people', 'unprofessional', 'truck']]

***** Please Note that the "important_words" have changed since the code has been re-run after the findings were interpreted in writing. The findings remain relevant because the same themes persist in the new output. *****

Theme 1 (Row 1) could be tied to “safety and worker conduct”. Stop words such as “rude”, “trailer”, “workers”, “never”, “security” suggest that reviews related to employee behavior and security interactions, often highlighting negative experiences (e.g., rude staff, poor treatment at entry points).

Theme 2 (Row 2) indicates “facility operations and logistics”. The keywords were “bad”, “facility”, “delivery”, “open”, and “staff”. These stop words focus on facility efficiency, including delivery processes and general operational concerns.

Theme 3 (Row 3) could highlight “entry/exit and security delays”. Stop words like “time”, “never”, “drivers”, “gate”, and “security” reveal long wait times and gate/security holdups. This appears to be especially true for truck drivers.

Theme 4 (Row 4) is speaking to the company’s “professionalism and work environment”. The keywords for this topic were “drop”, “trailer”, “take”, “hook”, and “unprofessional”. This likely reflects issues concerning unprofessional behavior and loading/unloading procedures.

Theme 5 (Row 5) addresses the “waiting experience and customer service quality”. The keywords - “long”, “waiting”, “driver”, “rude”, and “time”, establish a strong emphasis on long wait times combined with poor service interactions.